In [6]:
import numpy as np
import json
import logging
from pathlib import Path

# --- Configure logger ---
logger = logging.getLogger("deco.config")
logger.setLevel(logging.INFO)

# optional: add handler if not already attached
if not logger.handlers:
    handler = logging.StreamHandler()
    formatter = logging.Formatter("[%(asctime)s][%(levelname)s] %(message)s", "%H:%M:%S")
    handler.setFormatter(formatter)
    logger.addHandler(handler)

In [9]:
def load_deco_config(model_name: str):
    """Load DECO configuration for a specific model, with logging info."""
    base_path = Path("../configs/deco/base.json")
    model_cfg_path = Path(f"../configs/deco/{model_name}.json")

    # --- Load base config ---
    logger.info(f"Loading base DECO config from {base_path}")
    base = json.load(open(base_path))

    # --- Check if model-specific config exists ---
    if not model_cfg_path.exists():
        logger.warning(f"No model-specific DECO config found for '{model_name}', using base config only.")
        return base

    # --- Load override ---
    logger.info(f"Found override config: {model_cfg_path}")
    override = json.load(open(model_cfg_path))

    if "inherit" in override:
        logger.debug(f"Removing 'inherit' key from {model_cfg_path}")
        del override["inherit"]

    merged = {**base, **override}
    logger.info(f"✅ Loaded DECO config for model '{model_name}' "
                f"({len(override)} override keys, total {len(merged)} fields).")

    return merged


In [12]:

# Example
config = load_deco_config("gemma-3n-E4B-it")
print(config)

[08:10:47][INFO] Loading base DECO config from ../configs/deco/base.json
[08:10:47][INFO] Found override config: ../configs/deco/gemma-3n-E4B-it.json
[08:10:47][INFO] ✅ Loaded DECO config for model 'gemma-3n-E4B-it' (9 override keys, total 9 fields).


{'alpha': 0.6, 'threshold_top_p': 0.9, 'threshold_top_k': 20, 'entropy_threshold': 1.6, 'margin_threshold': 0.35, 'kl_threshold': 0.03, 'persistence': 2, 'alpha_schedule': {'type': 'entropy', 'schedule': {'low': 0.3, 'medium': 0.45, 'high': 0.65, 'very_high': 0.8}, 'entropy_ranges': {'low': [0.0, 0.8], 'medium': [0.8, 1.4], 'high': [1.4, 2.0], 'very_high': [2.0, 3.0]}, 'mode': 'adaptive'}, 'debug': False}
